In [1]:
!pip install py3Dmol requests

In [2]:
!pip install biopython tmtools numpy requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 7.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.2 MB/s eta 0:00:00
  Created wheel for tmtools: filename=tmtools-0.3.0-cp313-cp313-linux_x86_64.whl size=3271809 sha256=bc5d0970b381bb27946ecf05e8c89dcb8130229db10c8e54e5fbc4fcf02e36b9
  Stored in directory: /root/.cache/pip/wheels/4b/5a/d9/ac49c15b30b30c0c6af7b5608dcf494b2cd973661b657fc193
Successfully built tmtools


In [3]:
import requests
import os

def fetch_alphafold_structure(uniprot_id, output_dir="."):
    """Fetches a pre-computed PDB structure dynamically using the AlphaFold API."""

    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    print(f"Fetching metadata for {uniprot_id} from AlphaFold API...")

    #Query the API for the file URLs
    api_response = requests.get(api_url)
    if api_response.status_code == 200:
        data = api_response.json()
        if data and isinstance(data, list):
            pdb_url = data[0].get("pdbUrl")

            if pdb_url:
                print(f"Latest model found! Downloading from: {pdb_url}")
                pdb_response = requests.get(pdb_url)
                if pdb_response.status_code == 200:
                    file_path = os.path.join(output_dir, f"{uniprot_id}_AF.pdb")
                    with open(file_path, "wb") as f:
                        f.write(pdb_response.content)
                    print(f"Saved: {file_path}")
                    return file_path
                else:
                    print(f"Failed to download file from {pdb_url}. HTTP {pdb_response.status_code}")
                    return None
            else:
                print(f"API response missing 'pdbUrl' for {uniprot_id}.")
                return None
    else:
        print(f"API metadata request failed for {uniprot_id}. HTTP {api_response.status_code}")
        return None

if __name__ == "__main__":
    os.makedirs("structures", exist_ok=True)
    fetch_alphafold_structure("P69905", "structures")

Fetching metadata for P69905 from AlphaFold API...
Latest model found! Downloading from: https://alphafold.ebi.ac.uk/files/AF-P69905-F1-model_v6.pdb
Saved: structures/P69905_AF.pdb


In [4]:
import requests
import py3Dmol
import numpy as np

# Fetch the Data
sequence = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
response = requests.post(url, data=sequence)
pdb_data = response.text

# Extract Confidence Scores (pLDDT)
plddt_scores = []
for line in pdb_data.split('\n'):
    if line.startswith('ATOM') and line[13:15] == 'CA':
        score = float(line[60:66].strip())
        plddt_scores.append(score)

avg_plddt = np.mean(plddt_scores)
print(f"Prediction Complete!")
print(f"Sequence Length: {len(sequence)} amino acids")
print(f"Average Confidence Score (pLDDT): {avg_plddt:.2f} / 100")
if avg_plddt > .70:
    print("Assessment: High confidence prediction. Suitable for structural alignment.")
else:
    print("Assessment: Low confidence prediction. Use caution.")

view = py3Dmol.view(width=600, height=400)
view.addModel(pdb_data, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.show()

Prediction Complete!
Sequence Length: 51 amino acids
Average Confidence Score (pLDDT): 0.88 / 100
Assessment: High confidence prediction. Suitable for structural alignment.


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [5]:
import requests
import py3Dmol

# UniProt ID for Human Immunoglobulin gamma-1 heavy chain
uniprot_id = "P0DOX5"
url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
api_response = requests.get(url).json()
pdb_url = api_response[0].get("pdbUrl")
pdb_data = requests.get(pdb_url).text
print(f"Successfully fetched {uniprot_id} from AlphaFold DB!")
view = py3Dmol.view(width=600, height=400)
view.addModel(pdb_data, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.show()

Successfully fetched P0DOX5 from AlphaFold DB!


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
import os
import requests
from tmtools.io import get_structure, get_residue_data
from tmtools import tm_align
import os
import time
import requests

def predict_structure_esmfold(sequence, output_name, output_dir=".", max_retries=3):
    url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
    print(f"Predicting structure for {output_name} via ESMFold...")
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(url, data=sequence, timeout=30)
            if response.status_code == 200:
                file_path = os.path.join(output_dir, f"{output_name}_predicted.pdb")
                with open(file_path, "wb") as f:
                    f.write(response.content)
                print(f"Saved Prediction: {file_path}")
                return file_path
            elif response.status_code in [502, 503, 504]:
                wait_time = attempt * 3
                print(f"API Overloaded (HTTP {response.status_code}). Retrying in {wait_time}s (Attempt {attempt}/{max_retries})...")
                time.sleep(wait_time)

            else:
                print(f"Failed prediction. HTTP {response.status_code}: {response.text}")
                return None

        except requests.exceptions.RequestException as e:
            print(f"Network error: {e}. Retrying...")
            time.sleep(attempt * 3)

    print(f"Max retries ({max_retries}) reached. ESMFold API is currently unresponsive.")
    return None

def download_experimental_pdb(pdb_id, output_dir="."):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    print(f"Downloading experimental ground truth for {pdb_id}...")

    response = requests.get(url)
    if response.status_code == 200:
        file_path = os.path.join(output_dir, f"{pdb_id}_experimental.pdb")
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"Downloaded Truth: {file_path}")
        return file_path
    else:
        print(f"Failed to download experimental PDB {pdb_id}")
        return None
if __name__ == "__main__":
    os.makedirs("structures", exist_ok=True)
    pdb_id = "1UBQ"
    seq_1ubq = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
    predicted_file = predict_structure_esmfold(seq_1ubq, pdb_id, "structures")
    experimental_file = download_experimental_pdb(pdb_id, "structures")

    if predicted_file and experimental_file:
        print("\nCalculating structural alignment metrics...")
        struct_exp = get_structure(experimental_file)
        struct_pred = get_structure(predicted_file)
        chain_exp = next(struct_exp.get_chains())
        chain_pred = next(struct_pred.get_chains())
        coords_exp, seq_exp = get_residue_data(chain_exp)
        coords_pred, seq_pred = get_residue_data(chain_pred)
        res = tm_align(coords_pred, coords_exp, seq_pred, seq_exp)
        tm_score = res.tm_norm_chain2
        rmsd = res.rmsd
        print("\nStructural Evaluation Results")
        print(f"Target:   {pdb_id}")
        print(f"TM-score: {tm_score:.4f} / 1.0000")
        print(f"RMSD:     {rmsd:.4f} Å")
        print("-" * 35)

        if tm_score > 0.5:
            print("High Accuracy: The predicted structure shares the same global fold as the experimental truth.")
        elif tm_score > 0.17:
            print("Moderate Accuracy: There is structural similarity, but local variations exist.")
        else:
            print("Low Accuracy: The structures appear unrelated (Random fold).")

Predicting structure for 1UBQ via ESMFold...
Saved Prediction: structures/1UBQ_predicted.pdb
Downloaded Truth: structures/1UBQ_experimental.pdb

Calculating structural alignment metrics...

Structural Evaluation Results
Target:   1UBQ
TM-score: 0.9581 / 1.0000
RMSD:     0.8274 Å
-----------------------------------
High Accuracy: The predicted structure shares the same global fold as the experimental truth.


In [7]:
import requests
import py3Dmol
import numpy as np
import os

sequence = "MKQLEDKVEELLSKNYHLENEVARLKKLVGER"
output_name = "custom_sequence_prediction"
output_dir = "structures"
os.makedirs(output_dir, exist_ok=True)
predicted_file = predict_structure_esmfold(sequence, output_name, output_dir)
if predicted_file:
    with open(predicted_file, 'r') as f:
        pdb_data = f.read()
    plddt_scores = [float(line[60:66].strip()) for line in pdb_data.split('\n') if line.startswith('ATOM') and line[13:15] == 'CA']
    if plddt_scores:
        avg_plddt = np.mean(plddt_scores)
        print(f"Prediction Complete! Average pLDDT: {avg_plddt:.2f} / 1.00")
        view = py3Dmol.view(width=600, height=400)
        view.addModel(pdb_data, 'pdb')
        view.setStyle({'cartoon': {'color': 'spectrum'}})
        view.zoomTo()
        view.show()
    else:
        print("Error: Valid PDB format returned, but no C-alpha ATOM lines found.")
else:
    print("ESMFold prediction failed or was not saved.")

Predicting structure for custom_sequence_prediction via ESMFold...
Saved Prediction: structures/custom_sequence_prediction_predicted.pdb
Prediction Complete! Average pLDDT: 0.89 / 1.00


3Dmol.js failed to load for some reason. Please check your browser console for error messages.